# 50 — Phase 0 retrieval diagnostic

**Goal**: produce the bare-retriever baseline (BM25 / dense / fused wRRF) on the full dev split, with failure-mode breakdown and per-slice recall@20 tables.

Output drives the Phase 1 bundle prioritization in `documents/retrieval_improvement_plan.md` (see project memory `project_blind_a_first_results.md`).

**Hardware**: A100-40GB or RTX PRO 6000 Blackwell (95 GB). Runtime ~10–15 min on A100 (full dev, ~8000 turns, batch_size=64).

**What this notebook does**: wraps `scripts/phase0_retrieval_diagnostic.py` — all logic (TDD'd pure functions + orchestration) lives in the script. Cells: env setup → run script → render report inline.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth (the dense retriever pulls precomputed embeddings from the gated TalkPlay dataset).
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('NO HF_TOKEN — set it in Colab secrets (left sidebar > key icon) before running cell 5.', e)

# Optional: cache HF datasets/models on Drive to avoid re-download across sessions.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    print('HF_HOME =', os.environ['HF_HOME'])
except Exception as e:
    print('Drive not mounted (using ephemeral Colab cache).', e)

In [ ]:
# 4) Install deps. The diagnostic only needs retrieval-side libs (no training stack).
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml bm25s scipy

In [ ]:
# 5) (Optional) smoke test on 5 sessions before the full run, to catch wiring issues fast.
# Skip this if you've already verified the script works on this branch.
!python scripts/phase0_retrieval_diagnostic.py \
    --sample 5 --batch-size 16 \
    --out-json /tmp/phase0_smoke.json \
    --out-md /tmp/phase0_smoke.md
print('--- smoke report ---')
print(open('/tmp/phase0_smoke.md').read())

In [ ]:
# 6) Full dev run. Defaults to config/110-prorank-rerank-devset.yaml + full dev (~8000 turns).
# Wallclock estimate: ~10-15 min on A100, faster on Blackwell.
!python scripts/phase0_retrieval_diagnostic.py \
    --batch-size 64 \
    --out-json data/phase0_diagnostic.json \
    --out-md documents/phase0_baseline_report.md

In [ ]:
# 7) Render the markdown report inline + show the raw JSON for downstream comparison.
from IPython.display import Markdown, display
display(Markdown(open('documents/phase0_baseline_report.md').read()))
print('\n--- raw JSON (top-level keys + per-component recall@20) ---')
import json
summary = json.load(open('data/phase0_diagnostic.json'))
print('n_turns:', summary['n_turns'])
for comp, m in summary['per_component_metrics'].items():
    print(f"  {comp:8s} recall@20={m['recall@20']:.3f}  recall@100={m['recall@100']:.3f}  ndcg@20={m['ndcg@20']:.3f}")
print('failure_breakdown:', summary['failure_breakdown'])

In [ ]:
# 8) (Optional) commit + push the report so it's anchored on the branch.
# Only run this if you want to capture the baseline in git.
!git config user.email 'orrimoch@gmail.com'
!git config user.name 'Or Rimoch (Colab)'
!git add data/phase0_diagnostic.json documents/phase0_baseline_report.md
!git commit -m 'phase0: bare-retriever diagnostic baseline on full dev'
# !git push origin {BRANCH}   # uncomment when ready to push